# AgriShield Kaggle Core Baseline

This notebook prepares the first AI/data-processing core for AgriShield.

It is intentionally conservative: the model learns from transparent weak labels generated by the current rule engine. Treat the output as a reproducible baseline and decision-support prototype, not as production-grade disaster prediction.

Outputs are written to `/kaggle/working/agrishield_outputs/`.

## What This Notebook Does

1. Loads advisory-zone metadata and hazard observations.
2. If no Kaggle dataset is attached, creates a small sample Noru-style dataset.
3. Engineers flood, typhoon, exposure, and timing features.
4. Generates weak labels from the rule-based risk engine.
5. Trains a RandomForest risk-score model and severity classifier.
6. Exports metrics, predictions, model card, and `risk_export_for_app.json`.

In [ ]:
from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
INPUT_DIR = Path('/kaggle/input/agrishield-core')
OUTPUT_DIR = Path('/kaggle/working/agrishield_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPOSURE_VALUE = {'low': 0.28, 'medium': 0.62, 'high': 1.0}

NUMERIC_FEATURES = [
    'lat', 'lng', 'rice_area_ha', 'elevation_min_m', 'elevation_max_m',
    'forecast_rain_72h_mm', 'rainfall_anomaly_pct', 'soil_saturation_pct',
    'storm_distance_km', 'storm_wind_kmh', 'lead_time_hours',
    'floodplain_exposure_score', 'coastal_wind_exposure_score', 'drainage_stress_score',
    'rain_load', 'storm_proximity_index', 'month', 'hour'
]

CATEGORICAL_FEATURES = ['zone_id', 'current_admin', 'dominant_season']

print('Notebook ready')
print('Output directory:', OUTPUT_DIR)

In [ ]:
ADVISORY_ZONES_CSV = '''zone_id,zone_name,current_admin,legacy_admin,lat,lng,rice_area_ha,dominant_season,floodplain_exposure,coastal_wind_exposure,drainage_stress,elevation_min_m,elevation_max_m
da-nang-hoa-vang,Hoa Vang upland-irrigated belt,Da Nang City,"Former Hoa Vang district, Da Nang",16.026,108.120,3150,Summer-Autumn rice,medium,medium,medium,12,90
da-nang-dien-ban-hoi-an,Dien Ban - Hoi An lowland rice belt,Da Nang City,"Former Quang Nam: Dien Ban, Hoi An",15.891,108.334,7200,Winter-Spring and Summer-Autumn rice,high,high,medium,0,8
da-nang-thang-binh-coastal,Thang Binh coastal plain,Da Nang City,"Former Quang Nam: Thang Binh",15.678,108.378,9600,Summer-Autumn rice,high,high,high,1,15
da-nang-dai-loc-vu-gia,Dai Loc - Vu Gia river corridor,Da Nang City,"Former Quang Nam: Dai Loc",15.880,107.980,8100,Winter-Spring and Summer-Autumn rice,high,medium,high,5,40
hue-lagoon-rice,Hue lagoon rice and aquaculture edge,Hue City,Former Thua Thien Hue lagoon communes,16.490,107.630,6900,Winter-Spring rice,high,medium,medium,0,10
hue-phong-dien-buffer,Phong Dien inland buffer,Hue City,"Former Phong Dien district, Thua Thien Hue",16.580,107.390,5400,Summer-Autumn rice,medium,low,medium,8,55
'''

HAZARD_OBSERVATIONS_CSV = '''event_name,timestamp,zone_id,forecast_rain_72h_mm,rainfall_anomaly_pct,soil_saturation_pct,storm_distance_km,storm_wind_kmh,lead_time_hours,observed_impact_score
noru-2022,2022-09-25T04:00:00+07:00,da-nang-hoa-vang,78,18,50,520,70,72,
noru-2022,2022-09-25T04:00:00+07:00,da-nang-dien-ban-hoi-an,92,24,56,480,74,72,
noru-2022,2022-09-25T04:00:00+07:00,da-nang-thang-binh-coastal,98,26,58,455,78,72,
noru-2022,2022-09-25T04:00:00+07:00,da-nang-dai-loc-vu-gia,88,22,55,490,72,72,
noru-2022,2022-09-25T04:00:00+07:00,hue-lagoon-rice,70,16,48,610,66,72,
noru-2022,2022-09-25T04:00:00+07:00,hue-phong-dien-buffer,62,12,43,650,62,72,
noru-2022,2022-09-26T04:00:00+07:00,da-nang-hoa-vang,120,35,63,330,88,48,
noru-2022,2022-09-26T04:00:00+07:00,da-nang-dien-ban-hoi-an,145,48,70,275,96,48,
noru-2022,2022-09-26T04:00:00+07:00,da-nang-thang-binh-coastal,158,52,73,250,102,48,
noru-2022,2022-09-26T04:00:00+07:00,da-nang-dai-loc-vu-gia,150,50,72,300,94,48,
noru-2022,2022-09-26T04:00:00+07:00,hue-lagoon-rice,112,34,62,390,84,48,
noru-2022,2022-09-26T04:00:00+07:00,hue-phong-dien-buffer,96,28,56,430,78,48,
noru-2022,2022-09-27T04:00:00+07:00,da-nang-hoa-vang,185,58,76,190,108,24,
noru-2022,2022-09-27T04:00:00+07:00,da-nang-dien-ban-hoi-an,220,76,84,135,120,24,
noru-2022,2022-09-27T04:00:00+07:00,da-nang-thang-binh-coastal,238,84,88,118,128,24,
noru-2022,2022-09-27T04:00:00+07:00,da-nang-dai-loc-vu-gia,230,82,86,160,116,24,
noru-2022,2022-09-27T04:00:00+07:00,hue-lagoon-rice,175,60,75,245,98,24,
noru-2022,2022-09-27T04:00:00+07:00,hue-phong-dien-buffer,145,44,66,285,88,24,
noru-2022,2022-09-28T03:30:00+07:00,da-nang-hoa-vang,240,76,88,75,126,6,
noru-2022,2022-09-28T03:30:00+07:00,da-nang-dien-ban-hoi-an,285,98,94,32,142,6,
noru-2022,2022-09-28T03:30:00+07:00,da-nang-thang-binh-coastal,300,105,96,18,148,6,
noru-2022,2022-09-28T03:30:00+07:00,da-nang-dai-loc-vu-gia,295,102,97,60,134,6,
noru-2022,2022-09-28T03:30:00+07:00,hue-lagoon-rice,220,78,88,175,112,6,
noru-2022,2022-09-28T03:30:00+07:00,hue-phong-dien-buffer,180,58,76,220,96,6,
noru-2022,2022-09-29T04:00:00+07:00,da-nang-hoa-vang,165,42,82,210,55,0,
noru-2022,2022-09-29T04:00:00+07:00,da-nang-dien-ban-hoi-an,210,64,92,185,58,0,
noru-2022,2022-09-29T04:00:00+07:00,da-nang-thang-binh-coastal,225,70,94,170,60,0,
noru-2022,2022-09-29T04:00:00+07:00,da-nang-dai-loc-vu-gia,245,82,96,205,54,0,
noru-2022,2022-09-29T04:00:00+07:00,hue-lagoon-rice,155,44,82,300,48,0,
noru-2022,2022-09-29T04:00:00+07:00,hue-phong-dien-buffer,125,30,70,340,42,0,
'''

def load_tables():
    zones_path = INPUT_DIR / 'advisory_zones.csv'
    obs_path = INPUT_DIR / 'hazard_observations.csv'
    if zones_path.exists() and obs_path.exists():
        print('Loading attached Kaggle dataset:', INPUT_DIR)
        zones = pd.read_csv(zones_path)
        observations = pd.read_csv(obs_path)
    else:
        print('No attached Kaggle dataset found. Using embedded sample data.')
        zones = pd.read_csv(StringIO(ADVISORY_ZONES_CSV))
        observations = pd.read_csv(StringIO(HAZARD_OBSERVATIONS_CSV))
    return zones, observations

zones, observations = load_tables()
display(zones.head())
display(observations.head())

In [ ]:
def severity_from_score(score: float) -> str:
    if score >= 85:
        return 'severe'
    if score >= 65:
        return 'high'
    if score >= 40:
        return 'guarded'
    return 'low'

def exposure_value(level) -> float:
    return EXPOSURE_VALUE.get(str(level).strip().lower(), 0.28)

def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    data['timestamp'] = pd.to_datetime(data['timestamp'], utc=True)
    data['observed_impact_score'] = pd.to_numeric(data.get('observed_impact_score'), errors='coerce')

    data['floodplain_exposure_score'] = data['floodplain_exposure'].map(exposure_value)
    data['coastal_wind_exposure_score'] = data['coastal_wind_exposure'].map(exposure_value)
    data['drainage_stress_score'] = data['drainage_stress'].map(exposure_value)
    data['rain_load'] = (data['forecast_rain_72h_mm'] / 260).clip(0, 1)
    data['storm_proximity_index'] = (1 - data['storm_distance_km'] / 360).clip(0, 1)
    data['month'] = data['timestamp'].dt.month
    data['hour'] = data['timestamp'].dt.hour

    rain_load = (data['forecast_rain_72h_mm'] / 260).clip(0, 1)
    anomaly_load = (data['rainfall_anomaly_pct'] / 90).clip(0, 1)
    saturation_load = (data['soil_saturation_pct'] / 95).clip(0, 1)
    flood_exposure = data['floodplain_exposure_score'] * 0.7 + data['drainage_stress_score'] * 0.3

    wind_load = (data['storm_wind_kmh'] / 150).clip(0, 1)
    distance_load = (1 - data['storm_distance_km'] / 360).clip(0, 1)
    lead_time_load = (1 - np.maximum(data['lead_time_hours'] - 24, 0) / 96).clip(0.22, 1)

    data['weak_flood_score'] = np.round(
        100 * (rain_load * 0.38 + anomaly_load * 0.20 + saturation_load * 0.22 + flood_exposure * 0.20)
    ).clip(0, 100)
    data['weak_typhoon_score'] = np.round(
        100 * (wind_load * 0.36 + distance_load * 0.30 + lead_time_load * 0.14 + data['coastal_wind_exposure_score'] * 0.20)
    ).clip(0, 100)

    primary = data[['weak_flood_score', 'weak_typhoon_score']].max(axis=1)
    secondary = data[['weak_flood_score', 'weak_typhoon_score']].min(axis=1)
    data['weak_composite_score'] = np.round(primary * 0.72 + secondary * 0.28).clip(0, 100)

    has_observed = data['observed_impact_score'].notna()
    data['target_score'] = np.where(has_observed, data['observed_impact_score'].clip(0, 100), data['weak_composite_score'])
    data['label_source'] = np.where(has_observed, 'observed_impact_score', 'weak_rule_label')
    data['target_severity'] = data['target_score'].map(severity_from_score)
    return data

def augment_weak_label_data(frame: pd.DataFrame, copies: int = 8) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    augmented = [frame]
    for copy_index in range(copies):
        sample = frame.copy()
        sample['event_name'] = sample['event_name'].astype(str) + f'-aug-{copy_index + 1}'
        sample['forecast_rain_72h_mm'] = (sample['forecast_rain_72h_mm'] + rng.normal(0, 24, len(sample))).clip(0, 360)
        sample['rainfall_anomaly_pct'] = (sample['rainfall_anomaly_pct'] + rng.normal(0, 12, len(sample))).clip(-40, 140)
        sample['soil_saturation_pct'] = (sample['soil_saturation_pct'] + rng.normal(0, 8, len(sample))).clip(0, 100)
        sample['storm_distance_km'] = (sample['storm_distance_km'] + rng.normal(0, 34, len(sample))).clip(0, 800)
        sample['storm_wind_kmh'] = (sample['storm_wind_kmh'] + rng.normal(0, 12, len(sample))).clip(0, 180)
        sample['lead_time_hours'] = (sample['lead_time_hours'] + rng.normal(0, 8, len(sample))).clip(0, 96)
        sample['observed_impact_score'] = np.nan
        augmented.append(sample)
    return pd.concat(augmented, ignore_index=True)

panel = observations.merge(zones, on='zone_id', how='left', validate='many_to_one')
source_panel = add_engineered_features(panel)
training_panel = add_engineered_features(augment_weak_label_data(source_panel))

print('Source rows:', len(source_panel))
print('Training rows after augmentation:', len(training_panel))
display(training_panel[['zone_id', 'timestamp', 'weak_flood_score', 'weak_typhoon_score', 'weak_composite_score', 'target_severity']].head())

In [ ]:
train_df, test_df = train_test_split(
    training_panel,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=training_panel['target_severity'] if training_panel['target_severity'].nunique() > 1 else None,
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), NUMERIC_FEATURES),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ]
)

regressor = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', RandomForestRegressor(n_estimators=320, min_samples_leaf=2, random_state=RANDOM_SEED)),
    ]
)

classifier = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(n_estimators=320, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_SEED)),
    ]
)

feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES
regressor.fit(train_df[feature_columns], train_df['target_score'])
classifier.fit(train_df[feature_columns], train_df['target_severity'])

pred_score = regressor.predict(test_df[feature_columns]).clip(0, 100)
pred_severity = classifier.predict(test_df[feature_columns])

mae = float(np.mean(np.abs(test_df['target_score'].to_numpy() - pred_score)))
rmse = float(math.sqrt(np.mean((test_df['target_score'].to_numpy() - pred_score) ** 2)))
accuracy = float(accuracy_score(test_df['target_severity'], pred_severity))

metrics = {
    'backend': 'sklearn_random_forest',
    'rows': int(len(training_panel)),
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'score_metrics': {'mae': round(mae, 3), 'rmse': round(rmse, 3)},
    'severity_accuracy': round(accuracy, 3),
    'severity_report': classification_report(test_df['target_severity'], pred_severity, output_dict=True, zero_division=0),
    'label_source_counts': training_panel['label_source'].value_counts().to_dict(),
    'severity_counts': training_panel['target_severity'].value_counts().to_dict(),
}

print(json.dumps(metrics, indent=2))

In [ ]:
predictions = test_df[
    ['event_name', 'timestamp', 'zone_id', 'zone_name', 'target_score', 'target_severity', 'weak_flood_score', 'weak_typhoon_score', 'label_source']
].copy()
predictions['predicted_score'] = np.round(pred_score, 2)
predictions['predicted_severity'] = pred_severity

latest = source_panel.sort_values('timestamp').groupby('zone_id', as_index=False).tail(1).sort_values('zone_id').copy()
latest_pred = regressor.predict(latest[feature_columns]).clip(0, 100)

zones_export = []
for row, predicted in zip(latest.to_dict(orient='records'), latest_pred):
    score = float(round(predicted, 2))
    zones_export.append({
        'zoneId': row['zone_id'],
        'zoneName': row['zone_name'],
        'currentAdmin': row['current_admin'],
        'timestamp': row['timestamp'].isoformat(),
        'modelPredictedCompositeScore': score,
        'modelPredictedSeverity': severity_from_score(score),
        'weakCompositeScore': float(row['weak_composite_score']),
        'weakFloodScore': float(row['weak_flood_score']),
        'weakTyphoonScore': float(row['weak_typhoon_score']),
        'labelSource': row['label_source'],
    })

risk_export = {
    'generatedAt': datetime.now(timezone.utc).isoformat(),
    'modelPurpose': 'AgriShield district-scale advisory risk baseline',
    'claimBoundary': 'Decision-support indicator; no commune or village precision claimed.',
    'zones': zones_export,
}

model_card = f'''# AgriShield Risk Model Card

## Intended Use

District-scale advisory risk scoring for rice-focused climate resilience demos.

## Current Label Source

The default target is a weak rule label generated from transparent flood and typhoon scoring. If `observed_impact_score` is supplied, the pipeline uses that value as the target for those rows.

## Validation Snapshot

- Rows: {metrics['rows']}
- Train rows: {metrics['train_rows']}
- Test rows: {metrics['test_rows']}
- MAE: {metrics['score_metrics']['mae']}
- RMSE: {metrics['score_metrics']['rmse']}
- Severity accuracy: {metrics['severity_accuracy']}

## Limits

- This model is not an official warning system.
- The current sample data is small and weakly labeled.
- Production requires official forecast feeds, verified historical impacts, and local hydromet validation.
- No commune-level or village-level precision is claimed.
'''

predictions.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)
source_panel.to_csv(OUTPUT_DIR / 'processed_source_panel.csv', index=False)
(OUTPUT_DIR / 'training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'risk_export_for_app.json').write_text(json.dumps(risk_export, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'model_card.md').write_text(model_card, encoding='utf-8')

model_bundle = {
    'regressor': regressor,
    'classifier': classifier,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'created_at': datetime.now(timezone.utc).isoformat(),
}
dump(model_bundle, OUTPUT_DIR / 'agrishield_risk_model.joblib')

print('Artifacts written:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path)

display(pd.DataFrame(zones_export))

## Package Outputs For Kaggle Download

Run this cell at the end if Kaggle does not show the nested `agrishield_outputs` folder clearly. It creates one ZIP file and also copies the most important artifacts directly into `/kaggle/working/`.

In [ ]:
import shutil
from IPython.display import FileLink, display

IMPORTANT_ARTIFACTS = [
    'risk_export_for_app.json',
    'training_metrics.json',
    'predictions.csv',
    'model_card.md',
]

zip_base = Path('/kaggle/working/agrishield_outputs')
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('Created ZIP:', zip_path)

for name in IMPORTANT_ARTIFACTS:
    src = OUTPUT_DIR / name
    dst = Path('/kaggle/working') / name
    if src.exists():
        shutil.copy2(src, dst)
        print('Copied:', dst)
    else:
        print('Missing expected artifact:', src)

print('Root /kaggle/working files:')
for path in sorted(Path('/kaggle/working').glob('*')):
    print('-', path)

display(FileLink('/kaggle/working/agrishield_outputs.zip'))
for name in IMPORTANT_ARTIFACTS:
    file_path = Path('/kaggle/working') / name
    if file_path.exists():
        display(FileLink(str(file_path)))


## Next Step

Download `risk_export_for_app.json`, `training_metrics.json`, `predictions.csv`, and `model_card.md` from Kaggle output. The JSON export can be wired back into the Next.js demo once the model snapshot is good enough to show.